In [12]:
import os
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.cluster.hierarchy import linkage, fcluster
from collections import defaultdict

base_dir = "/home/local/ASUAD/jrtandel/transchema/autopipeline-benchmarks/github-pipelines"

# 1) Gather schemas for each “length3_*” case
case_list = sorted(
    d for d in os.listdir(base_dir)
    if d.startswith("length") and os.path.isdir(os.path.join(base_dir, d))
)
schemas = {}
for case in list(case_list):  # list() because we might remove on error
    try:
        path = os.path.join(base_dir, case)
        files = [f for f in os.listdir(path) if (f.startswith("test") or f=="target.csv") and f.endswith(".csv")]
        cols = set()
        for f in files:
            cols |= set(pd.read_csv(os.path.join(path, f), nrows=0).columns)
        if not cols:
            raise ValueError("no columns")
        schemas[case] = cols
    except Exception:
        case_list.remove(case)

# 2) Binarize and cluster
schema_lists = [list(schemas[c]) for c in case_list]
X = MultiLabelBinarizer().fit_transform(schema_lists)
Z = linkage(X, method="average", metric="jaccard")

# 3) Assign flat clusters (no plotting!)
threshold = 0.5
cluster_ids = fcluster(Z, t=threshold, criterion="distance")
clusters = defaultdict(list)
for case, cid in zip(case_list, cluster_ids):
    clusters[cid].append(case)

# ── 4) Extract each case’s target.csv schema, dropping Unnamed:index cols ──
target_schemas = {}
for case in case_list:
    tgt_path = os.path.join(base_dir, case, "target.csv")
    if not os.path.isfile(tgt_path):
        target_schemas[case] = None
        continue

    # Read only header
    df = pd.read_csv(tgt_path, nrows=0)

    # Filter out any 'Unnamed:' columns (usually index columns)
    cols = [c for c in df.columns if not c.startswith("Unnamed")]
    target_schemas[case] = frozenset(cols)
from collections import defaultdict

# … after you’ve built `clusters` (cid → [cases]) and `target_schemas` …

# ── Filter out clusters smaller than 5 ──────────────────────────────────────
filtered_clusters = {cid: members
                     for cid, members in clusters.items()
                     if len(members) >= 15}

# ── Sort remaining clusters by size (largest first) ─────────────────────────
cluster_items = sorted(filtered_clusters.items(),
                       key=lambda x: len(x[1]),
                       reverse=True)

for cid, members in cluster_items:
    print(f"\nCluster {cid} (size={len(members)}):")
    print(f"  All cases: {sorted(members)}")
    grouping = defaultdict(list)
    for case in members:
        grouping[target_schemas.get(case)].append(case)
    for schema, cases in grouping.items():
        cols = sorted(schema) if schema else []
        print(f"  Target schema ({len(cols)} cols): {cols}")
        print(f"    Cases: {sorted(cases)}")


Cluster 93 (size=59):
  All cases: ['length1_26', 'length1_60', 'length1_69', 'length1_7', 'length2_10', 'length2_11', 'length2_13', 'length2_16', 'length2_18', 'length2_22', 'length2_23', 'length2_25', 'length2_30', 'length2_31', 'length2_32', 'length2_34', 'length2_38', 'length2_44', 'length2_5', 'length2_56', 'length2_58', 'length2_62', 'length2_63', 'length2_69', 'length2_72', 'length2_73', 'length2_75', 'length2_76', 'length2_77', 'length2_79', 'length2_81', 'length2_84', 'length2_88', 'length2_89', 'length2_9', 'length2_92', 'length2_98', 'length2_99', 'length3_28', 'length3_29', 'length3_44', 'length3_45', 'length3_46', 'length3_50', 'length3_51', 'length3_69', 'length3_70', 'length3_71', 'length3_72', 'length3_73', 'length3_74', 'length3_75', 'length3_76', 'length3_77', 'length4_72', 'length4_73', 'length4_83', 'length5_90', 'length5_94']
  Target schema (6 cols): ['city', 'date', 'driver_count', 'fare', 'ride_id', 'type']
    Cases: ['length1_26', 'length1_69', 'length1_7']
 